In [ ]:
%%capture
%pip install langchain chromadb sentence-transformers beautifulsoup4 requests langchain langchain-community langchain-huggingface langchain-chroma faiss-cpu youtube-transcript-api

In [13]:
import requests
from bs4 import BeautifulSoup
import time
import json
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.formatters import JSONFormatter

*Das HF_TOKEN muss mit `setx HF_TOKEN "TOKENWERT"` gesetzt werden*

In [3]:
def fetch_threads_list(category_url):
    resp = requests.get(category_url + ".json")
    data = resp.json()
    threads = []
    for t in data["topic_list"]["topics"]:
        threads.append({
            "topic_id": t["id"],
            "topic_slug": t["slug"],
            "title": t["title"]
        })
    return threads

def fetch_thread_posts(topic_id, topic_slug):
    url = f"https://forums.forza.net/t/{topic_slug}/{topic_id}.json"
    resp = requests.get(url)
    data = resp.json()
    posts = []
    for p in data["post_stream"]["posts"]:
        posts.append({
            "post_id": p["id"],
            "author_name": p["name"],
            "username": p["username"],
            "created_at": p["created_at"],
            "content_text": BeautifulSoup(p["cooked"], "html.parser").get_text(),
            "topic_id": p["topic_id"],
            "topic_slug": p["topic_slug"]
        })
    return posts

CATEGORY_URL = "https://forums.forza.net/tags/c/community-hub/ugc/tuning/148/fh5"
threads = fetch_threads_list(CATEGORY_URL)

all_posts = []
for t in threads:
    all_posts.extend(fetch_thread_posts(t["topic_id"], t["topic_slug"]))


In [24]:
with open("../data/rag_text/chunked_docs.json", "w", encoding="utf-8") as f:
    json.dump(all_posts, f, ensure_ascii=False, indent=4)

In [ ]:
video_id = "aq96SH5zy3Y" # TODO add multiple video ids
ytt_api = YouTubeTranscriptApi()
formatter = JSONFormatter()

transcript = ytt_api.fetch(video_id)

json_formatted = formatter.format_transcript(transcript)
data = json.loads(json_formatted)

texts = [entry["text"] for entry in data]
full_text = " ".join(texts)

# prepare JSON object for each video
output = {
    "video_id": video_id,
    "text": full_text
}

with open("../data/rag_text/yt_transcripts.json", 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)